# Step-1: Setting the Dependencies and necessities

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

Unsloth helps builders create custom models better & faster. We're developing the all in one solution to help you create highly-accurate custom models 30x faster with 90% less memory use

**Now that Unsloth is installed, we can proceed to load our model and tokenizer.**

#Step 2: Load the Model and Tokenizer


we will load the DeepSeek model using Unsloth’s optimized methods. I am using the DeepSeek-R1-Distill-Llama-8B model.

In [ ]:
from unsloth import FastLanguageModel
import torch


max_seq_length = 2048
dtype = None
load_in_4bit = True  # Enable 4-bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.2.4: Fast Llama patching. Transformers: 4.48.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.9k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

If you want to fine-tune another model, just change the model_name field.

#Step 3: Apply LoRA Adapters for Efficient Fine-Tuning


Low-Rank Adaptation (LoRA) allows us to fine-tune only a small subset of the model’s parameters, making training faster and memory efficient.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (controls low-rank approximation quality)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Layers to apply LoRA
    lora_alpha=16, # Scaling factor for LoRA weights
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)


Unsloth 2025.2.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


#Step 4: Prepare the Training Dataset

In [ ]:
from datasets import load_dataset


dataset = load_dataset("Sulav/mental_health_counseling_conversations_sharegpt", split="train")

README.md:   0%|          | 0.00/624 [00:00<?, ?B/s]

(…)-00000-of-00001-3193672bedc3e534.parquet:   0%|          | 0.00/4.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Now we need to convert the dataset from ShareGPT style (“from”, ”value”) to Hugging face generic format(“role”, “content”).

In [ ]:
from unsloth.chat_templates import standardize_sharegpt


dataset = standardize_sharegpt(dataset)

Standardizing format:   0%|          | 0/3512 [00:00<?, ? examples/s]

#Step 5: Format Prompts

In [ ]:
from unsloth.chat_templates import get_chat_template

# Apply the Llama-3.1 chat template to the tokenizer
tokenizer = get_chat_template(
    tokenizer,  # Tokenizer being used
    chat_template="llama-3.1",  # The chat template format
)

# Function to format the conversation data into tokenized text
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'tokenizer' is not defined

To understand how conversations are rendered in Llama-3.1 format, you can print out an item in both its original conversation format and formatted text format

In [ ]:
# Print an item in its original conversation format
print(dataset[0]["conversations"])

# Print the same item in its formatted text format
print(dataset[0]["text"])

#Step 6: Set Up and Configure the Trainer

Now, we will configure the fine-tuning process using Hugging Face’s SFTTrainer. It automates key tasks like tokenization, batching, and optimization, making fine-tuning easier. SFTTrainer works efficiently with Unsloth, reducing VRAM usage and speeding up training.

I have limited the fine-tuning to 75 steps to speed things up, but you can set num_train_epochs=1 for a full run, and max_steps=None.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported


# Define training configurations
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=2,  # Number of examples per GPU batch
        gradient_accumulation_steps=4,  # Accumulate gradients over 4 batches before updating model
        warmup_steps=5,  # Number of warmup steps for learning rate schedule
        max_steps=75,  # Limit training steps to 75 (for quick testing)
        # num_train_epochs=1
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,  # Log training metrics after every step
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",  # Linear decay of learning rate
        seed=3407,
        output_dir="outputs",  # Directory to save model checkpoints
        report_to="none",  # Use this for WandB etc

    ),
)

Map (num_proc=2):   0%|          | 0/3512 [00:00<?, ? examples/s]

#Step 7: Train Only on Assistant Responses

To improve training efficiency, we will focus only on the assistant’s responses rather than user inputs.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",  # Mark user input
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",  # Mark assistant response
)
# Start training the model
trainer_stats = trainer.train()

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 3,512 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 75
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,3.415800
2,3.075000
3,3.246600
4,3.394700
5,3.137400
6,3.013500
7,2.590600
8,2.922000
9,2.832500
10,2.819500


The reduction of training loss here is a bit less because we have only fine-tuned the model for 150
 steps. For better results, it is recommended to train your dataset for 2-3 epochs on a large dataset and 3-5 epochs on a small dataset. Aim for at least 500+ steps, but if resources allow, training for 1000+ steps can further improve model performance.

#Step 8: Inference

After fine-tuning, we can use the trained model for inference to generate responses

In [ ]:
tokenizer = get_chat_template(
   tokenizer,
   chat_template = "llama-3.1",
)

tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
   {"role": "user", "content": "Make a joke on chinese president"}]
# Tokenize the user input with the chat template
inputs = tokenizer.apply_chat_template(
   messages,
   tokenize=True,
   add_generation_prompt=True,
   return_tensors="pt",
   padding=True,  # Add padding to match sequence lengths
).to("cuda")

attention_mask = inputs != tokenizer.pad_token_id

outputs = model.generate(
   input_ids=inputs,
   attention_mask=attention_mask,
   max_new_tokens=64,
   use_cache=True,  # Use cache for faster token generation
   temperature=0.6,  # Controls randomness in responses
   min_p=0.1,
)


text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)


NameError: name 'get_chat_template' is not defined

#Step 9: Saving the Model & Tokenizer


In [ ]:
my_model="MindSeek-8B-T1"
model.save_pretrained(my_model)
tokenizer.save_pretrained(my_model)

NameError: name 'model' is not defined

# Step 10: Push it to huggingface

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [1]:
# from huggingface_hub import HfApi, HfFolder


# my_model = "MindSeek-8B-T1"
# repo_id = f"tushaa/{my_model}"

# model.save_pretrained(my_model)
# tokenizer.save_pretrained(my_model)


# model.push_to_hub(repo_id)
# tokenizer.push_to_hub(repo_id)

In [ ]:
!zip -r /content/MindSeek-8B-T1.zip /content/MindSeek-8B-T1

In [ ]:
from google.colab import drive
!mkdir -p /content/gdrive
drive.mount('/content/gdrive')

!zip -r /content/gdrive/MyDrive/MindSeek-8B-T1.zip /content/MindSeek-8B-T1.zip

In [ ]:
!pip install gradio torch transformers